# GRPO mini-training on math problems

A self-contained, runnable implementation of Group Relative Policy Optimization (GRPO)
on a small set of math problems — the same algorithm used in DeepSeek-R1.

**What this notebook does:**

| Step | What happens |
| ---- | ------------ |
| 0 | Load `Qwen/Qwen3-0.6B-Base` as both policy and frozen reference |
| 1 | Load math problems from `rasbt/math_full_minus_math500` (or fall back to built-in samples) |
| 2 | Inspect one GRPO step in detail — rollouts, rewards, advantages |
| 3 | Run a mini training loop (10–20 steps, CPU-feasible) |
| 4 | Compare base model vs GRPO-fine-tuned model on held-out questions |

> **Runtime note:** Full GRPO trains for thousands of steps on GPUs. This notebook runs
> 15–20 steps on CPU to illustrate the mechanics. Reward improvement is small — the
> learning curve and rollout behaviour are the point.

Read alongside `src/content/concepts/reasoning-models/grpo.md`.

## 0. Setup

```bash
# in .venv-qwen
pip install datasets
```

Kernel: **Qwen demo** (`.venv-qwen`). See `notebooks/reasoning-models/inference-time-scaling.ipynb` §0 for setup.

In [12]:
import copy
import re
import json
import random
from collections import defaultdict

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

print("torch", torch.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

torch 2.14.0+cpu
device: cpu


## 1. Load model (policy) and frozen reference copy

In [13]:
MODEL_NAME = "Qwen/Qwen3-0.6B-Base"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading policy model...")
policy = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32, device_map="auto"
)
policy.train()

# Frozen reference — stays fixed throughout training
print("Creating frozen reference copy...")
ref_model = copy.deepcopy(policy)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

print(f"Policy params: {sum(p.numel() for p in policy.parameters()) / 1e6:.0f}M")

Loading tokenizer...
Loading policy model...


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1747.90it/s]


Creating frozen reference copy...
Policy params: 596M


## 2. Load math problems

We use `rasbt/math_full_minus_math500` — the MATH training set with the MATH-500 test
problems removed, so we can evaluate cleanly on MATH-500 later.

We filter to **Level 1–2** problems (easiest) to get any learning signal in a small
number of steps on CPU.

In [14]:
# Simple arithmetic problems — short, unambiguous, solvable in < 120 tokens.
# These give the model a real chance to earn format + accuracy rewards in 20 steps on CPU.
# The MATH dataset problems (GCD of 7!, arithmetic sequences, etc.) require multi-step
# reasoning that a base model can't reliably do in 120 tokens, causing all-zero rewards
# and no learning signal.
BUILT_IN_PROBLEMS = [
    {"problem": "What is 15 + 27?", "answer": "42"},
    {"problem": "A bag has 5 red and 3 blue marbles. How many marbles total?", "answer": "8"},
    {"problem": "Ram has 3 chocolates, Sam has 5. They each eat 1. How many are left?", "answer": "6"},
    {"problem": "If a train travels 60 km/h for 2 hours, how far does it go?", "answer": "120"},
    {"problem": "What is 144 divided by 12?", "answer": "12"},
    {"problem": "A rectangle is 8 cm wide and 5 cm tall. What is its area?", "answer": "40"},
    {"problem": "What is 7 squared?", "answer": "49"},
    {"problem": "If you have 100 and spend 37, how much is left?", "answer": "63"},
    {"problem": "What is the perimeter of a square with side 9?", "answer": "36"},
    {"problem": "How many seconds in 3 minutes?", "answer": "180"},
    {"problem": "What is 8 times 6?", "answer": "48"},
    {"problem": "A class has 30 students. 12 are girls. How many are boys?", "answer": "18"},
    {"problem": "What is 250 divided by 5?", "answer": "50"},
    {"problem": "A hexagon has how many sides?", "answer": "6"},
    {"problem": "What is 3 cubed?", "answer": "27"},
    {"problem": "If apples cost 3 each and you buy 7, how much do you pay?", "answer": "21"},
    {"problem": "What is 19 + 43?", "answer": "62"},
    {"problem": "A triangle has angles 60, 70 and x degrees. What is x?", "answer": "50"},
    {"problem": "What is 5 factorial?", "answer": "120"},
    {"problem": "How many minutes in 4 hours?", "answer": "240"},
    {"problem": "What is 9 times 9?", "answer": "81"},
    {"problem": "A dozen eggs minus 5. How many are left?", "answer": "7"},
    {"problem": "What is the square root of 64?", "answer": "8"},
    {"problem": "If you run 5 km each day for 6 days, how far in total?", "answer": "30"},
    {"problem": "What is 200 minus 73?", "answer": "127"},
]

# Use built-in problems by default — they give reliable reward signal in a short run.
# To use the MATH dataset instead, set USE_MATH_DATASET = True (requires internet access).
USE_MATH_DATASET = False

if USE_MATH_DATASET:
    try:
        from datasets import load_dataset
        ds = load_dataset("rasbt/math_full_minus_math500", split="train")
        easy = [
            {"problem": r["problem"], "answer": r["answer"]}
            for r in ds
            if r.get("level") in ("Level 1", "Level 2")
            and r.get("answer")
            and re.fullmatch(r"-?\d+(?:\.\d+)?", str(r["answer"]).strip())
        ]
        if len(easy) >= 20:
            random.seed(42)
            problems = random.sample(easy, 30)
            print(f"Loaded {len(problems)} Level-1/2 numeric problems from rasbt/math_full_minus_math500")
        else:
            raise ValueError("too few easy numeric problems")
    except Exception as e:
        print(f"Dataset load failed ({e}), falling back to built-in problems.")
        problems = BUILT_IN_PROBLEMS
else:
    problems = BUILT_IN_PROBLEMS
    print(f"Using {len(problems)} built-in arithmetic problems.")

train_problems = problems[:20]
eval_problems  = problems[20:] if len(problems) > 20 else problems[:5]

print(f"Train: {len(train_problems)}  Eval: {len(eval_problems)}")
print("\nSample problem:", train_problems[0]["problem"])
print("Answer:", train_problems[0]["answer"])

Using 25 built-in arithmetic problems.
Train: 20  Eval: 5

Sample problem: What is 15 + 27?
Answer: 42


## 3. GRPO building blocks

### 3a. Prompt template and reward functions

In [15]:
def make_prompt(problem: str) -> str:
    """Wrap problem in a CoT prompt that requests <think> tags."""
    return (
        f"Problem: {problem}\n"
        "Think step by step inside <think>...</think> tags, "
        "then write the final number after Answer:\n"
        "<think>"
    )


def extract_final_answer(text: str) -> str | None:
    """Parse the number after 'Answer:' or the last number in the text."""
    m = re.search(r"Answer\s*:\s*(-?\d+(?:\.\d+)?)", text, re.I)
    if m:
        return m.group(1).strip()
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else None


def reward_format(text: str) -> float:
    """1.0 if the response uses <think>...</think> structure, else 0.0."""
    return 1.0 if re.search(r"<think>.*?</think>", text, re.S) else 0.0


def reward_accuracy(text: str, gold: str) -> float:
    """1.0 if the parsed answer matches gold (numeric comparison), else 0.0."""
    pred = extract_final_answer(text)
    if pred is None:
        return 0.0
    try:
        return 1.0 if abs(float(pred) - float(gold)) < 1e-6 else 0.0
    except ValueError:
        return 1.0 if pred.strip() == gold.strip() else 0.0


def total_reward(text: str, gold: str) -> float:
    """Format reward + accuracy reward — same as DeepSeek-R1-Zero."""
    return reward_format(text) + reward_accuracy(text, gold)


# Quick sanity check
sample_text = "<think>3+5=8; 8-2=6</think>\nAnswer: 6"
print("format:", reward_format(sample_text))
print("accuracy:", reward_accuracy(sample_text, "6"))
print("total:", total_reward(sample_text, "6"))

format: 1.0
accuracy: 1.0
total: 2.0


### 3b. Generate rollouts with per-token log-probs

We need log-probs of generated tokens to compute both the policy gradient loss
and the KL divergence penalty.

In [16]:
def generate_rollouts(
    model,
    prompt: str,
    n: int = 4,
    max_new_tokens: int = 150,
    temperature: float = 0.8,
    top_k: int = 40,
    top_p: float = 0.9,
):
    """
    Sample `n` rollouts for `prompt`.
    Returns list of (text, generated_ids, token_log_probs) per rollout.
    token_log_probs: Tensor [T] of per-token log-probs under `model`.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            num_return_sequences=n,
            pad_token_id=tokenizer.eos_token_id,
            return_dict_in_generate=True,
            output_scores=True,
        )

    rollouts = []
    for i, seq in enumerate(out.sequences):
        gen_ids = seq[prompt_len:]
        text = tokenizer.decode(gen_ids, skip_special_tokens=True)
        # Collect step-wise log-probs for generated tokens
        lps = []
        for t, step_logits in enumerate(out.scores):
            if t >= len(gen_ids):
                break
            tid = int(gen_ids[t].item())
            lp = F.log_softmax(step_logits[i], dim=-1)[tid].item()
            lps.append(lp)
            if tid == tokenizer.eos_token_id:
                break
        rollouts.append((text, gen_ids[:len(lps)], torch.tensor(lps)))
    return rollouts


def get_logprobs_for_ids(model, prompt: str, gen_ids: torch.Tensor) -> torch.Tensor:
    """
    Re-score generated token ids under `model` (used for reference log-probs).
    Returns Tensor [T] of log-probs.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    full_ids = torch.cat([inputs["input_ids"][0], gen_ids.to(model.device)])
    with torch.no_grad():
        logits = model(full_ids.unsqueeze(0)).logits[0]  # [L, vocab]
    # logits[t] predicts token at t+1; for generated tokens starting at prompt_len:
    gen_logits = logits[prompt_len - 1 : prompt_len - 1 + len(gen_ids)]
    lps = F.log_softmax(gen_logits, dim=-1)
    return lps[torch.arange(len(gen_ids)), gen_ids.to(model.device)]


print("Helpers defined.")

Helpers defined.


### 3c. GRPO loss

In [17]:
def grpo_loss(
    policy_lps: list[torch.Tensor],   # one Tensor[T_i] per rollout
    ref_lps: list[torch.Tensor],      # same shape, from frozen ref model
    rewards: torch.Tensor,            # [N] scalar reward per rollout
    beta: float = 0.04,
) -> torch.Tensor:
    """
    GRPO loss = policy gradient (group-relative advantages) + KL penalty.

    Advantage: A_i = (r_i − mean(r)) / std(r)   — group normalised, no critic needed

    KL approx (non-negative, same as TRL / DeepSeek-R1):
        KL(π_ref || π_θ) ≈ exp(log π_ref − log π_θ) − (log π_ref − log π_θ) − 1
    This is always ≥ 0 (it equals 0 iff π_θ == π_ref) so the penalty
    only ever pushes the policy *toward* the reference, never away from it.
    The naive (log π_θ − log π_ref) estimate can go negative when the policy
    assigns lower probability to generated tokens than the reference did,
    which causes the optimizer to actively make things worse.
    """
    # --- group-relative advantages ---
    adv = (rewards - rewards.mean()) / (rewards.std() + 1e-8)  # [N]

    pg_terms, kl_terms = [], []
    for i, (plps, rlps) in enumerate(zip(policy_lps, ref_lps)):
        # policy gradient: -A_i * mean_t log π_θ(t)
        pg_terms.append(-adv[i] * plps.mean())
        # Non-negative KL approximation (per-token, averaged)
        log_ratio = rlps.detach() - plps          # log π_ref − log π_θ
        kl_approx = torch.exp(log_ratio) - log_ratio - 1   # ≥ 0
        kl_terms.append(kl_approx.mean())

    pg_loss = torch.stack(pg_terms).mean()
    kl_loss = torch.stack(kl_terms).mean()

    return pg_loss + beta * kl_loss, pg_loss.detach(), kl_loss.detach()


print("grpo_loss defined.")

grpo_loss defined.


## 4. Inspect one GRPO step in detail

Before training, walk through a single step to see rollouts, rewards, advantages.

In [18]:
INSPECT_PROBLEM = train_problems[0]
prompt = make_prompt(INSPECT_PROBLEM["problem"])
gold   = INSPECT_PROBLEM["answer"]

print("Problem:", INSPECT_PROBLEM["problem"])
print("Gold answer:", gold)
print()

N_ROLLOUTS = 5
rollouts = generate_rollouts(policy, prompt, n=N_ROLLOUTS, max_new_tokens=120)

rewards = torch.tensor([total_reward(text, gold) for text, _, _ in rollouts])
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-8)

print(f"{'#':>2}  {'Reward':>6}  {'Adv':>6}  {'Ans':>6}  Trace (first 120 chars)")
print("-" * 80)
for i, (text, _, _) in enumerate(rollouts):
    ans = extract_final_answer(text)
    snippet = text.replace("\n", " ")[:120]
    print(f"{i+1:>2}  {rewards[i].item():>6.1f}  {advantages[i].item():>+6.2f}  {str(ans):>6}  {snippet}")

print()
print(f"mean_reward={rewards.mean():.3f}  std={rewards.std():.3f}")

Problem: What is 15 + 27?
Gold answer: 42

 #  Reward     Adv     Ans  Trace (first 120 chars)
--------------------------------------------------------------------------------
 1     0.0   -0.73      27   15 + 27 = ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ ♁ �
 2     0.0   -0.73    None  ... ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ ◆ �
 3     1.0   +1.10      42   คำตอบ: 15 + 27 = 42
 4     1.0   +1.10      42  : 15 + 27 StartElement: 15 StartElement: 27 StartElement: 42
 5     0.0   -0.73    None  ... ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ �

mean_reward=0.400  std=0.548


## 5. Mini GRPO training loop

15–20 steps, one problem per step, `N = 4` rollouts each.
On CPU this takes ~5–15 min depending on your machine.

Reward won't jump dramatically in 20 steps — watch the **trend** and the logged rollouts.

In [19]:
GRPO_STEPS     = 20      # raise to 100+ on GPU
N_PER_STEP     = 4       # rollouts per question
MAX_NEW_TOKENS = 120
LEARNING_RATE  = 5e-6
BETA_KL        = 0.04    # KL penalty coefficient

optimizer = torch.optim.AdamW(
    [p for p in policy.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
)

history = []  # (step, mean_reward, pg_loss, kl_loss)

for step in range(GRPO_STEPS):
    prob = train_problems[step % len(train_problems)]
    prompt = make_prompt(prob["problem"])
    gold   = prob["answer"]

    # 1. Sample rollouts (no grad)
    rollouts = generate_rollouts(
        policy, prompt,
        n=N_PER_STEP,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    # 2. Compute rewards
    rewards = torch.tensor([total_reward(t, gold) for t, _, _ in rollouts])

    # Skip if all rollouts have the same reward (std=0 → advantages=0 → pg_loss=0,
    # and only the KL term would act, which can corrupt the model).
    if rewards.std() < 1e-6:
        mean_r = rewards.mean().item()
        history.append((step + 1, mean_r, 0.0, 0.0))
        if (step + 1) % 5 == 0 or step == 0:
            print(
                f"step {step+1:3d}  mean_reward={mean_r:.3f}  SKIPPED (no variance)  "
                f"answers={[extract_final_answer(t) for t,_,_ in rollouts]}"
            )
        continue

    # 3. Re-score under policy (with grad) and ref (no grad)
    policy_lps, ref_lps = [], []
    for text, gen_ids, _ in rollouts:
        # policy log-probs with grad
        inputs = tokenizer(prompt, return_tensors="pt").to(policy.device)
        plen   = inputs["input_ids"].shape[1]
        full   = torch.cat([inputs["input_ids"][0], gen_ids.to(policy.device)])
        logits = policy(full.unsqueeze(0)).logits[0]
        gen_logits = logits[plen - 1 : plen - 1 + len(gen_ids)]
        lps = F.log_softmax(gen_logits, dim=-1)
        policy_lps.append(lps[torch.arange(len(gen_ids)), gen_ids.to(policy.device)])

        # reference log-probs without grad
        ref_lps.append(get_logprobs_for_ids(ref_model, prompt, gen_ids))

    # 4. GRPO loss
    optimizer.zero_grad()
    loss, pg_loss, kl_loss = grpo_loss(policy_lps, ref_lps, rewards, beta=BETA_KL)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    optimizer.step()

    mean_r = rewards.mean().item()
    history.append((step + 1, mean_r, pg_loss.item(), kl_loss.item()))

    if (step + 1) % 5 == 0 or step == 0:
        print(
            f"step {step+1:3d}  "
            f"mean_reward={mean_r:.3f}  "
            f"pg={pg_loss.item():+.4f}  "
            f"kl={kl_loss.item():.5f}  "
            f"answers={[extract_final_answer(t) for t,_,_ in rollouts]}"
        )

print("\nDone.")

step   1  mean_reward=0.500  pg=+0.0215  kl=0.00000  answers=['27', '42', '15', '42']
step   5  mean_reward=1.000  SKIPPED (no variance)  answers=['12', '12', '12', '12']
step  10  mean_reward=0.750  pg=+0.0069  kl=0.13523  answers=['180', '180', '3', '180']
step  15  mean_reward=1.000  SKIPPED (no variance)  answers=['27', '27', '27', '27']
step  20  mean_reward=1.000  SKIPPED (no variance)  answers=['240', '240', '240', '240']

Done.


## 6. Training curve

In [20]:
steps_h   = [h[0] for h in history]
rewards_h = [h[1] for h in history]
kl_h      = [h[3] for h in history]

# Simple text plot (no matplotlib needed)
MAX_BAR = 40
print("Mean reward per step (bar = proportion of max 2.0)")
print("-" * 60)
for s, r in zip(steps_h, rewards_h):
    bar = int(r / 2.0 * MAX_BAR)
    print(f"step {s:3d}  {r:.3f}  {'|' * bar}")

print()
print("KL divergence from reference:")
for s, k in zip(steps_h, kl_h):
    print(f"  step {s:3d}  kl={k:.6f}")

Mean reward per step (bar = proportion of max 2.0)
------------------------------------------------------------
step   1  0.500  ||||||||||
step   2  1.000  ||||||||||||||||||||
step   3  0.500  ||||||||||
step   4  0.500  ||||||||||
step   5  1.000  ||||||||||||||||||||
step   6  0.750  |||||||||||||||
step   7  1.000  ||||||||||||||||||||
step   8  1.000  ||||||||||||||||||||
step   9  0.750  |||||||||||||||
step  10  0.750  |||||||||||||||
step  11  0.750  |||||||||||||||
step  12  0.750  |||||||||||||||
step  13  1.000  ||||||||||||||||||||
step  14  0.750  |||||||||||||||
step  15  1.000  ||||||||||||||||||||
step  16  1.000  ||||||||||||||||||||
step  17  1.000  ||||||||||||||||||||
step  18  1.000  ||||||||||||||||||||
step  19  1.000  ||||||||||||||||||||
step  20  1.000  ||||||||||||||||||||

KL divergence from reference:
  step   1  kl=0.000000
  step   2  kl=0.000000
  step   3  kl=0.030968
  step   4  kl=0.076638
  step   5  kl=0.000000
  step   6  kl=0.048297
  step   7  k

## 7. Before vs after comparison

Compare the base (reference) model against the GRPO-fine-tuned policy on eval problems.

**Metrics:**
- Accuracy rate (correct final answer)
- Format rate (uses `<think>` tags)
- Mean total reward

In [21]:
def evaluate(model, problems, n_samples: int = 3, max_new_tokens: int = 150, label: str = "model"):
    """Evaluate model on a list of problems. Returns metrics dict."""
    correct, formatted, total_rew, n = 0, 0, 0.0, 0
    details = []
    model.eval()
    for prob in problems:
        prompt = make_prompt(prob["problem"])
        gold   = prob["answer"]
        rollouts = generate_rollouts(model, prompt, n=n_samples, max_new_tokens=max_new_tokens)
        for text, _, _ in rollouts:
            rew  = total_reward(text, gold)
            acc  = reward_accuracy(text, gold)
            fmt  = reward_format(text)
            correct   += acc
            formatted += fmt
            total_rew += rew
            n         += 1
            details.append({
                "problem": prob["problem"],
                "gold": gold,
                "pred": extract_final_answer(text),
                "reward": rew,
                "text": text,
            })
    model.train()
    return {
        "accuracy": correct / n,
        "format_rate": formatted / n,
        "mean_reward": total_rew / n,
        "n": n,
        "details": details,
    }


print("Evaluating BASE (reference) model...")
base_metrics = evaluate(ref_model, eval_problems, n_samples=3, label="base")

print("Evaluating GRPO-fine-tuned policy...")
grpo_metrics = evaluate(policy, eval_problems, n_samples=3, label="grpo")

print()
print(f"{'Metric':<18} {'Base':>10} {'GRPO':>10}")
print("-" * 40)
for key in ["accuracy", "format_rate", "mean_reward"]:
    print(f"{key:<18} {base_metrics[key]:>10.3f} {grpo_metrics[key]:>10.3f}")
print(f"{'samples':18} {base_metrics['n']:>10} {grpo_metrics['n']:>10}")

Evaluating BASE (reference) model...
Evaluating GRPO-fine-tuned policy...

Metric                   Base       GRPO
----------------------------------------
accuracy                0.467      0.800
format_rate             0.000      0.000
mean_reward             0.467      0.800
samples                    15         15


## 8. Side-by-side qualitative examples

In [22]:
N_SHOW = min(3, len(eval_problems))

for prob in eval_problems[:N_SHOW]:
    prompt = make_prompt(prob["problem"])
    gold   = prob["answer"]

    base_out = generate_rollouts(ref_model, prompt, n=1, max_new_tokens=150)[0][0]
    grpo_out = generate_rollouts(policy,    prompt, n=1, max_new_tokens=150)[0][0]

    print("=" * 70)
    print("PROBLEM:", prob["problem"])
    print("GOLD:", gold)
    print()
    print("BASE  → answer:", extract_final_answer(base_out),
          f"  reward={total_reward(base_out, gold):.1f}")
    print(base_out[:300])
    print()
    print("GRPO  → answer:", extract_final_answer(grpo_out),
          f"  reward={total_reward(grpo_out, gold):.1f}")
    print(grpo_out[:300])
    print()

PROBLEM: What is 9 times 9?
GOLD: 81

BASE  → answer: None   reward=0.0
.  ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ ⚊ �

GRPO  → answer: 81   reward=1.0
 9 times 9 equals 81.

PROBLEM: A dozen eggs minus 5. How many are left?
GOLD: 7

BASE  → answer: None   reward=0.0
… ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ ◍ �

GRPO  → answer: 7   reward=1.0
 a dozen eggs is 12.ultureInfo ultureInfo minus 5 eggs gives us the final number: 12 - 5 = 7.

PROBLEM: What is the square root of 64?
GOLD: 8

BASE  → answer: 8   reward=1.0

Tags: square root, square root 64
Solution: Step 1: To find the square root of 64, we need to determine a number that, when multiplied by itself, equals 64.
Step 2: We can start by estimating the square root. Since 8 * 8 = 64, the square root of 64 is 8.
Step 3: To confirm that 8 is indeed the squa

GRPO  → answer: 8   reward=1.0
 64 is the square of 8, so the squ

## 9. Key observations

| What to look for | Why it matters |
| ---------------- | -------------- |
| **Format rate** rises after GRPO | Model learns `<think>…</think>` structure from the format reward signal alone |
| **KL stays small and non-negative** | The β penalty is working correctly — policy has not drifted from the reference |
| **Steps marked SKIPPED** | All rollouts had the same reward (zero variance) so advantages = 0; skipping prevents the KL term from corrupting the model |
| Accuracy improvement is modest (20 steps, CPU) | Expected — full DeepSeek-R1-Zero training runs thousands of steps on 671B params with 8+ GPUs |
| Advantages centre around 0 | Normalisation by group mean/std keeps gradient scale stable without a critic |

### Implementation notes

**KL formula matters.** The naive estimate `mean(log π_θ − log π_ref)` can go negative
when the policy assigns lower probability to generated tokens than the reference did.
A negative KL × positive β = negative penalty term; gradient descent then makes KL
*more* negative, actively corrupting the model. The correct non-negative approximation is:

```
KL ≈ exp(log π_ref − log π_θ) − (log π_ref − log π_θ) − 1   ≥ 0
```

This equals 0 iff π_θ = π_ref and grows positively for any divergence — the penalty
now always pulls the policy back toward the reference, never away from it.

The mechanics are identical to what happens at scale — the reward signal, advantage
normalisation, and KL penalty all work the same way. Only the number of steps and the
model size differ.

See `src/content/concepts/reasoning-models/grpo.md` for the full conceptual walkthrough.